# 8장 (실습 1): 컨테이너 이미지를 직접 만들고, CVE를 찾아 보안 Gate 판정해보기

> 책 『AI 보안, 클라우드에서 시작한다』 8장 [실습] 1번을 노트북에서 단계별로 실행하는 버전입니다. 작은 AI 서빙 앱을 컨테이너 이미지로 빌드하고, Trivy로 CVE를 스캔하고 SBOM을 만든 뒤, CISA KEV와 FIRST EPSS로 표 8-10 기준의 "차단/경고/통과"를 판정합니다. 소요 시간은 40~60분입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. 이미지의 취약점이 **베이스 이미지와 직접 고른 라이브러리 중 어디에서** 오는지 스캔 결과와 SBOM으로 확인합니다.
2. CVSS 등급 순서가 아니라 **실제 악용(KEV)과 악용 확률(EPSS)** 로 판정하면 우선순위가 어떻게 바뀌는지 확인합니다.
3. 판정 결과를 **종료 코드**로 돌려주는 보안 Gate를 만들고, 베이스 이미지와 의존성을 갱신해 다시 판정합니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | 예정 |
| 1 | 실습 앱과 두 가지 Dockerfile 살펴보기 | 예정 |
| 2 | v1 이미지 빌드 | 예정 |
| 3 | CVE 스캔과 SBOM 생성 | 예정 |
| 4 | KEV와 EPSS로 판정하기 | 예정 |
| 5 | v2로 다시 판정 | 예정 |
| 6 | 파이프라인에 넣기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

## 실습 시나리오

```text
 Dockerfile.old ─build─▶ ai-serving:v1 ─trivy─▶ scan-v1.json ─┐
                                      └─trivy cyclonedx─▶ sbom-v1.json
 CISA KEV(실제 악용된 CVE) ──────────────────────────────────┼─▶ gate.py ─▶ 표 8-10 판정 ─▶ 종료 코드 1(배포 불가) / 0(가능)
 FIRST EPSS(30일 내 악용 확률) ──────────────────────────────┘
 Dockerfile(베이스 이미지·의존성 갱신) ─build─▶ ai-serving:v2 ─▶ 다시 스캔·판정
```

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Docker와 Trivy가 필요하고, KEV·EPSS 조회를 위해 인터넷 연결이 필요합니다. 설치는 [SETUP.md](../SETUP.md)를 따릅니다.
>
> **2. 노트북 열기**<br>
> 처음이라면 [SETUP.md](../SETUP.md)의 6단계(VS Code와 Jupyter 노트북 연결)를 먼저 한 번 해 둡니다. 그다음에는 아래 두 방법 가운데 하나로 이 노트북을 엽니다.
> - **VS Code**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs && code .`를 실행합니다. 창 왼쪽 아래에 `WSL: Ubuntu-24.04`가 보이면, 왼쪽 탐색기에서 `ch08-devsecops-image-gate` 폴더를 펼쳐 이 노트북을 엽니다. 그리고 오른쪽 위 커널 표시를 눌러 **Select Another Kernel...** → **Python Environments...** → `.venv-lab`을 고릅니다. 목록에 `.venv-lab`이 없으면 6단계 ⑥의 2번처럼 `Python: Select Interpreter` → **Enter interpreter path...**로 `~/.venv-lab/bin/python`의 위치를 먼저 알려 줍니다.
> - **웹 브라우저(Jupyter Lab)**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs/ch08-devsecops-image-gate && ~/.venv-lab/bin/jupyter lab`을 실행하고, 화면에 나온 `http://localhost:8888/...` 주소를 Windows의 브라우저에서 엽니다.
> - 노트북 파일만 따로 내려받거나 다른 폴더로 옮기지 마세요. 노트북은 자기가 들어 있는 `ch08-devsecops-image-gate` 폴더의 실습 파일을 읽습니다. 위 두 방법처럼 저장소 안에서 열면 따로 신경 쓸 것은 없습니다.
>
> **3. 기준 버전**<br>
> - 아래 출력은 2026년 10월 23일 Ubuntu 24.04(WSL2), Docker 29.1.3, Trivy 0.76.0, Python 3.12에서 실행한 결과입니다. 시각, 번호, ID 같은 값은 실행할 때마다 다릅니다. 직접 실행할 때는 출력이 없는 `ch08_lab1_image_cve_gate.ipynb`을 쓰세요.

> 📌 (참고) CVE와 KEV·EPSS 정보는 매일 바뀝니다. 이 노트북의 숫자는 실행한 날의 값이며, 책과 README의 숫자(2026년 10월 17일)와 조금 다를 수 있습니다. 표는 모두 셀이 실행 시점에 직접 계산합니다.

## 0. 실습 준비

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 실습 앱과 두 가지 Dockerfile 살펴보기 | 예정 |
| 2 | v1 이미지 빌드 | 예정 |
| 3 | CVE 스캔과 SBOM 생성 | 예정 |
| 4 | KEV와 EPSS로 판정하기 | 예정 |
| 5 | v2로 다시 판정 | 예정 |
| 6 | 파이프라인에 넣기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

In [1]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "docker": "docker --version",
    "trivy": "trivy --version | head -1",
    "python3": "python3 --version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

docker     ✅ Docker version 29.1.3, build 29.1.3-0ubuntu3~24.04.2
trivy      ✅ Version: 0.76.0
python3    ✅ Python 3.12.3


실습 파일을 `~/image-lab`으로 복사합니다. 리눅스 홈에서 작업하는 편이 빌드가 훨씬 빠르고, 스캔 결과 파일도 저장소를 어지럽히지 않습니다.

In [2]:
import shutil
import sys

WORK = Path.home() / "image-lab"
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(LAB / "image-lab", WORK)
print(sorted(p.name for p in WORK.iterdir()))

['Dockerfile', 'Dockerfile.old', 'app.py', 'gate.py', 'requirements-old.txt', 'requirements.txt']


## 1. 실습 앱과 두 가지 Dockerfile 살펴보기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 실습 앱과 두 가지 Dockerfile 살펴보기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | v1 이미지 빌드 | 예정 |
| 3 | CVE 스캔과 SBOM 생성 | 예정 |
| 4 | KEV와 EPSS로 판정하기 | 예정 |
| 5 | v2로 다시 판정 | 예정 |
| 6 | 파이프라인에 넣기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

`app.py`는 질문을 받아 답하는 흉내만 내는 FastAPI 앱입니다. 모델은 없지만, 실제 AI 서빙 이미지와 마찬가지로 **베이스 이미지(OS 패키지) + Python 런타임 + 오픈소스 라이브러리 + 우리 코드**로 이루어집니다.

| 파일 | 베이스 이미지 | 의존성 | 의도 |
|---|---|---|---|
| `Dockerfile.old` (v1) | `python:3.9.0-slim` (2020년, Debian 10) | `requirements-old.txt` (2021년 무렵 버전) | 오래 방치된 이미지 재현 |
| `Dockerfile` (v2) | `python:3.12-slim` (지원 중, Debian 13) | `requirements.txt` (갱신된 버전) | 베이스 이미지와 의존성 갱신, 비루트 사용자 |

두 Dockerfile과 두 의존성 목록의 차이를 나란히 봅니다.

In [3]:
import difflib


def diff(a, b):
    lines = difflib.unified_diff((WORK / a).read_text().splitlines(), (WORK / b).read_text().splitlines(),
                                 a, b, lineterm="", n=0)
    print("\n".join(l for l in lines if not l.startswith("@@")))


diff("Dockerfile.old", "Dockerfile")
print()
diff("requirements-old.txt", "requirements.txt")

--- Dockerfile.old
+++ Dockerfile
-# 오래된 베이스 이미지 + 오래된 의존성 (일부러 취약하게 만든 v1)
-# 실습 외의 용도로 쓰지 말 것. Trivy로 스캔하면 OS 패키지와 Python 라이브러리 양쪽에서 취약점이 많이 나온다.
-# 지원이 끝난 Python 3.9.0 기반 이미지 → OS 패키지도 함께 오래되었다
-FROM python:3.9.0-slim
+# 지원 중인 베이스 이미지 + 갱신된 의존성 (v2)
+# Dockerfile.old와 비교하면 바뀐 곳은 세 군데다: 베이스 이미지, requirements 파일, USER nobody
+FROM python:3.12-slim
-# 수년 전 버전으로 고정된 라이브러리 목록 (requirements-old.txt)
-COPY requirements-old.txt .
-RUN pip install --no-cache-dir -r requirements-old.txt
+# 의존성 목록을 먼저 복사해 설치한다 (코드만 바뀌었을 때는 이 단계를 캐시로 건너뛰어 빌드가 빨라진다)
+COPY requirements.txt .
+# --no-cache-dir: 설치 파일 캐시를 이미지에 남기지 않아 크기를 줄인다
+RUN pip install --no-cache-dir -r requirements.txt
-# USER 지정이 없다 → root로 실행된다 (v2에서는 USER nobody로 고쳤다)
+# root가 아닌 사용자로 실행 → 앱이 뚫려도 컨테이너 안에서 할 수 있는 일이 줄어든다
+USER nobody

--- requirements-old.txt
+++ requirements.txt
-fastapi==0.65.2
-uvicorn==0.13.4
-requests==2.25.1
-urllib3==1.26.4
-jinja2==2.11.3
-pyyaml==5.3.1
-pillow==8.2.0
-aiohttp==3.8.1
+fastapi==0.115.6
+uvicorn==0.3

## 2. v1 이미지 빌드

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 실습 앱과 두 가지 Dockerfile 살펴보기 | ✅ 완료 |
| 2 | v1 이미지 빌드 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | CVE 스캔과 SBOM 생성 | 예정 |
| 4 | KEV와 EPSS로 판정하기 | 예정 |
| 5 | v2로 다시 판정 | 예정 |
| 6 | 파이프라인에 넣기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

v1 이미지를 빌드합니다. 처음에는 베이스 이미지를 내려받고 의존성을 설치하느라 몇 분 걸립니다.

In [4]:
%%bash
cd ~/image-lab
docker build --progress=plain -f Dockerfile.old -t ai-serving:v1 . 2>&1 | grep -E '^#[0-9]+ \[1/5\] FROM|naming to' | sed 's/^#[0-9]* //'
docker images ai-serving --format 'table {{.Repository}}:{{.Tag}}\t{{.Size}}'

[1/5] FROM docker.io/library/python:3.9.0-slim@sha256:de8d4a338fb815509de2046cea4ff48959c84aabc1b65c

b41c5e2da1ac599b04
naming to docker.io/library/ai-serving:v1 done


REPOSITORY:TAG   SIZE
ai-serving:v1    237MB


빌드 로그의 `[1/5] FROM ...python:3.9.0-slim@sha256:...`에 주목합니다. 태그(`3.9.0-slim`)는 바뀔 수 있지만 다이제스트(`sha256:...`)는 이미지 내용 그 자체를 가리킵니다. 운영 파이프라인에서는 베이스 이미지를 다이제스트로 고정하고, 갱신할 때만 바꿉니다.

## 3. CVE 스캔과 SBOM 생성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 실습 앱과 두 가지 Dockerfile 살펴보기 | ✅ 완료 |
| 2 | v1 이미지 빌드 | ✅ 완료 |
| 3 | CVE 스캔과 SBOM 생성 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | KEV와 EPSS로 판정하기 | 예정 |
| 5 | v2로 다시 판정 | 예정 |
| 6 | 파이프라인에 넣기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

### 3.1. 스캔 결과 합계

Trivy의 사람용 표는 길기 때문에 대상별 합계만 봅니다. 처음 실행하면 취약점 DB를 내려받느라 조금 걸립니다.

In [5]:
%%bash
cd ~/image-lab
trivy image --severity HIGH,CRITICAL ai-serving:v1 2>/dev/null | grep -E '^(ai-serving|Python|Total)'

ai-serving:v1 (debian 10.6)
Total: 126 (HIGH: 100, CRITICAL: 26)
Python (python-pkg)
Total: 65 (HIGH

: 57, CRITICAL: 8)


### 3.2. JSON 결과와 SBOM 만들기

판정 스크립트와 파이썬 분석에 쓸 JSON 결과와, 이미지 구성요소 목록인 SBOM(CycloneDX)을 만듭니다.

In [6]:
%%bash
cd ~/image-lab
trivy image --quiet --severity HIGH,CRITICAL --format json -o scan-v1.json ai-serving:v1
trivy image --quiet --format cyclonedx -o sbom-v1.json ai-serving:v1
ls -l scan-v1.json sbom-v1.json | awk '{print $5, $NF}'

208560 sbom-v1.json
1319311 scan-v1.json


### 3.3. 취약점은 어디에서 왔나

스캔 결과의 `Results`는 대상(OS 패키지, Python 패키지)마다 나뉩니다. 대상별로 HIGH·CRITICAL 건수를 셉니다.

In [7]:
from collections import Counter


def by_target(scan_file):
    scan = json.loads((WORK / scan_file).read_text())
    rows = []
    for res in scan.get("Results", []):
        vulns = res.get("Vulnerabilities") or []
        sev = Counter(v["Severity"] for v in vulns)
        kind = "OS 패키지" if res["Class"] == "os-pkgs" else "Python 패키지"
        rows.append((kind, res["Target"][:45], len(vulns), sev["CRITICAL"], len({v["PkgName"] for v in vulns})))
    return rows


table(["구분", "대상", "HIGH 이상", "CRITICAL", "취약한 패키지 수"], by_target("scan-v1.json"))

| 구분 | 대상 | HIGH 이상 | CRITICAL | 취약한 패키지 수 |
|---|---|---|---|---|
| OS 패키지 | ai-serving:v1 (debian 10.6) | 126 | 26 | 34 |
| Python 패키지 | Python | 65 | 8 | 9 |

결과를 보면 취약점의 대부분이 **우리가 직접 고른 라이브러리가 아니라 베이스 이미지의 OS 패키지**에서 나왔습니다. 개발자는 `requirements-old.txt`만 신경 썼겠지만, `FROM` 한 줄이 이만큼을 끌고 들어왔습니다. 8-5에서 "실제 배포물 기준"으로 SBOM을 만들어야 한다고 한 이유입니다.

SBOM을 열어 보면 Python 패키지뿐 아니라 `openssl`, `libc6`, `zlib1g` 같은 OS 패키지까지 모두 구성요소로 기록되어 있습니다.

In [8]:
sbom = json.loads((WORK / "sbom-v1.json").read_text())
comps = sbom["components"]
kinds = Counter(next((p["value"] for p in c.get("properties", []) if p["name"] == "aquasecurity:trivy:PkgType"), c["type"])
                for c in comps)
print(f"SBOM 형식: {sbom['bomFormat']} {sbom['specVersion']}, 구성요소 {len(comps)}개")
table(["구성요소 종류", "개수"], kinds.most_common())
names = {c["name"]: c.get("version", "") for c in comps}
table(["예시 구성요소", "버전"], [(n, names[n]) for n in ["openssl", "libc6", "zlib1g", "pillow", "fastapi"] if n in names])

SBOM 형식: CycloneDX 1.7, 구성요소 122개


| 구성요소 종류 | 개수 |
|---|---|
| debian | 93 |
| python-pkg | 28 |
| operating-system | 1 |

| 예시 구성요소 | 버전 |
|---|---|
| openssl | 1.1.1d-0+deb10u3 |
| libc6 | 2.28-10 |
| zlib1g | 1:1.2.11.dfsg-1 |
| fastapi | 0.65.2 |

## 4. KEV와 EPSS로 판정하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 실습 앱과 두 가지 Dockerfile 살펴보기 | ✅ 완료 |
| 2 | v1 이미지 빌드 | ✅ 완료 |
| 3 | CVE 스캔과 SBOM 생성 | ✅ 완료 |
| 4 | KEV와 EPSS로 판정하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | v2로 다시 판정 | 예정 |
| 6 | 파이프라인에 넣기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

### 4.1. 판정 스크립트

`gate.py`는 Trivy 결과의 CVE마다 ① CISA KEV 목록에 있는지, ② EPSS 점수가 얼마인지 조회하고, ③ 표 8-10 기준으로 판정합니다. 차단이 하나라도 있으면 종료 코드 1을 돌려줍니다.

In [9]:
src = (WORK / "gate.py").read_text()
print(src[src.index("def decide"):src.index("rows = []")])

def decide(cve, v):
    """취약점 하나의 판정과 이유를 돌려준다. 위에서부터 차례로 검사해 처음 맞는 판정을 쓴다."""
    fixed = bool(v.get("FixedVersion"))
    if cve in kev:
        return "차단", "KEV 등재 (실제 악용 확인)"           # 심각도와 관계없이 무조건 막는다
    if v["Severity"] == "CRITICAL" and epss.get(cve, 0) >= 0.1:
        return "차단", "CRITICAL + EPSS 높음"                # 치명적이고 악용 확률도 10% 이상
    if not fixed:
        # 고칠 버전이 아직 없으면 막아도 해결되지 않는다 → 영향 여부를 VEX로 기록하고 운영 단계에서 지켜본다
        return "통과(VEX)", "수정 버전 없음 — VEX 작성, 9-3 모니터링으로 인계"
    return "경고", "수정 버전 있음 — 다음 릴리스까지 조치"




In [10]:
%%bash
cd ~/image-lab
python3 gate.py scan-v1.json > gate-v1.txt; echo "종료 코드: $?"
head -8 gate-v1.txt
tail -1 gate-v1.txt

종료 코드: 1


차단 | CVE-2023-4863 | HIGH | 1.0 | True | Pillow | 8.2.0 | 10.0.1 | KEV 등재 (실제 악용 확

인)
차단 | CVE-2021-3711 | CRITICAL | 0.878 | False | openssl | 1.1.1d-0+deb10u3 | 1.1.1d-0+deb10

u7 | CRITICAL + EPSS 높음
차단 | CVE-2019-8457 | CRITICAL | 0.454 | False | libdb5.3 | 5.3.28+df

sg1-0.5 | - | CRITICAL + EPSS 높음
차단 | CVE-2022-25236 | CRITICAL | 0.353 | False | libexpat1 

...

있음 — 다음 릴리스까지 조치
경고 | CVE-2024-2961 | HIGH | 0.883 | False | libc6 | 2.2

8-10 | 2.28-10+deb10u3 | 수정 버전 있음 — 다음 릴리스까지 조치
경고 | CVE-2022-12

92 | HIGH | 0.826 | False | openssl | 1.1.1d-0+deb10u3 | 1.1.1n-0+deb10u2 | 수정 버전 있음 —

 다음 릴리스까지 조치


총 146건, 차단 5건 → 배포 불가


### 4.2. 판정 결과 집계

`gate.py`의 출력(한 줄에 `판정 | CVE | 등급 | EPSS | KEV | 패키지 | 설치 버전 | 수정 버전 | 사유`)을 읽어 판정별로 셉니다.

In [11]:
def gate_rows(txt):
    rows = []
    for line in (WORK / txt).read_text().splitlines():
        f = [x.strip() for x in line.split(" | ")]
        if len(f) == 9:
            rows.append({"verdict": f[0], "cve": f[1], "sev": f[2], "epss": float(f[3]), "kev": f[4] == "True",
                         "pkg": f[5], "fixed": f[7], "reason": f[8]})
    return rows


g1 = gate_rows("gate-v1.txt")
table(["판정", "건수"], Counter(r["verdict"] for r in g1).most_common())
table(["판정", "CVE", "등급", "EPSS", "KEV", "패키지", "사유"],
      [(r["verdict"], r["cve"], r["sev"], r["epss"], "✔" if r["kev"] else "", r["pkg"], r["reason"])
       for r in g1 if r["verdict"] == "차단"])

| 판정 | 건수 |
|---|---|
| 경고 | 126 |
| 통과(VEX) | 15 |
| 차단 | 5 |

| 판정 | CVE | 등급 | EPSS | KEV | 패키지 | 사유 |
|---|---|---|---|---|---|---|
| 차단 | CVE-2023-4863 | HIGH | 1.0 | ✔ | Pillow | KEV 등재 (실제 악용 확인) |
| 차단 | CVE-2021-3711 | CRITICAL | 0.878 |  | openssl | CRITICAL + EPSS 높음 |
| 차단 | CVE-2019-8457 | CRITICAL | 0.454 |  | libdb5.3 | CRITICAL + EPSS 높음 |
| 차단 | CVE-2022-25236 | CRITICAL | 0.353 |  | libexpat1 | CRITICAL + EPSS 높음 |
| 차단 | CVE-2022-37434 | CRITICAL | 0.192 |  | zlib1g | CRITICAL + EPSS 높음 |

> 📌 (참고) EPSS 차단 기준(0.1 = 30일 내 악용 확률 10%)은 예시입니다. 조직의 위험 허용 수준에 맞게 정합니다. 표 8-10의 "인터넷 노출 구성요소" 조건은 이미지 정보만으로는 알 수 없으므로, 실무에서는 8-1의 외부 노출 구성요소 목록과 결합합니다.

### 4.3. CVSS 순서와 실제 위험 순서

CVSS 등급 순서(CRITICAL 먼저)로 줄을 세웠다면 KEV에 등재된 CVE는 몇 번째였을까요? 두 순서를 비교합니다.

In [12]:
sev_rank = {"CRITICAL": 0, "HIGH": 1}
by_cvss = sorted(g1, key=lambda r: (sev_rank[r["sev"]], r["cve"]))
by_risk = sorted(g1, key=lambda r: ({"차단": 0, "경고": 1, "통과(VEX)": 2}[r["verdict"]], -r["epss"]))
watch = [r for r in g1 if r["kev"]] + sorted([r for r in g1 if r["sev"] == "HIGH" and r["verdict"] == "경고"],
                                             key=lambda r: -r["epss"])[:1]
table(["CVE", "패키지", "등급", "EPSS", "KEV", "CVSS 순서", "KEV·EPSS 판정 순서", "판정"],
      [(r["cve"], r["pkg"], r["sev"], r["epss"], "✔" if r["kev"] else "",
        f"{by_cvss.index(r) + 1}번째", f"{by_risk.index(r) + 1}번째", r["verdict"]) for r in watch])

| CVE | 패키지 | 등급 | EPSS | KEV | CVSS 순서 | KEV·EPSS 판정 순서 | 판정 |
|---|---|---|---|---|---|---|---|
| CVE-2023-4863 | Pillow | HIGH | 1.0 | ✔ | 93번째 | 1번째 | 차단 |
| CVE-2022-2068 | openssl | HIGH | 0.964 |  | 68번째 | 6번째 | 경고 |

- **KEV에 등재된 CVE**는 CVSS 등급이 HIGH라 CVSS 순서로는 CRITICAL 뒤로 밀리지만, 실제 공격에 쓰였기 때문에 판정에서는 맨 위로 올라옵니다(책의 예: CVE-2023-4863, Pillow의 libwebp).
- **"경고"로 판정된 HIGH 등급 가운데 EPSS가 가장 높은 CVE**는 30일 안에 악용될 확률이 매우 높게 예측되는데도 "경고"에 머뭅니다. 판정 규칙이 "CRITICAL + EPSS"만 차단하기 때문입니다. 과제 1에서 규칙을 고쳐 봅니다.

## 5. v2로 다시 판정

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 실습 앱과 두 가지 Dockerfile 살펴보기 | ✅ 완료 |
| 2 | v1 이미지 빌드 | ✅ 완료 |
| 3 | CVE 스캔과 SBOM 생성 | ✅ 완료 |
| 4 | KEV와 EPSS로 판정하기 | ✅ 완료 |
| 5 | v2로 다시 판정 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 파이프라인에 넣기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

In [13]:
%%bash
cd ~/image-lab
docker build -q -t ai-serving:v2 . >/dev/null && echo "ai-serving:v2 빌드 완료"
trivy image --quiet --severity HIGH,CRITICAL --format json -o scan-v2.json ai-serving:v2
python3 gate.py scan-v2.json > gate-v2.txt; echo "종료 코드: $?"
tail -1 gate-v2.txt

ai-serving:v2 빌드 완료


종료 코드: 0


총 61건, 차단 0건 → 배포 가능


In [14]:
table(["구분", "대상", "HIGH 이상", "CRITICAL", "취약한 패키지 수"], by_target("scan-v2.json"))
table(["판정", "건수"], Counter(r["verdict"] for r in gate_rows("gate-v2.txt")).most_common())

| 구분 | 대상 | HIGH 이상 | CRITICAL | 취약한 패키지 수 |
|---|---|---|---|---|
| OS 패키지 | ai-serving:v2 (debian 13.7) | 44 | 0 | 17 |
| Python 패키지 | Python | 53 | 4 | 4 |

| 판정 | 건수 |
|---|---|
| 경고 | 53 |
| 통과(VEX) | 8 |

### 5.1. 두 이미지 비교

두 버전의 결과를 표 실습8-1 형식으로 모읍니다.

In [15]:
def summary(ver):
    t = {kind: (n, c) for kind, _, n, c, _ in by_target(f"scan-{ver}.json")}
    g = Counter(r["verdict"] for r in gate_rows(f"gate-{ver}.txt"))
    os_n, os_c = t.get("OS 패키지", (0, 0))
    py_n, py_c = t.get("Python 패키지", (0, 0))
    return (f"`ai-serving:{ver}`", f"{os_n} (CRITICAL {os_c})", f"{py_n} (CRITICAL {py_c})", sum(g.values()),
            g["차단"], g["경고"], g["통과(VEX)"], "불가" if g["차단"] else "가능")


table(["이미지", "OS 패키지 HIGH 이상", "Python 패키지 HIGH 이상", "판정 대상(중복 제거)", "차단", "경고", "통과(VEX)", "배포 판정"],
      [summary("v1"), summary("v2")])

| 이미지 | OS 패키지 HIGH 이상 | Python 패키지 HIGH 이상 | 판정 대상(중복 제거) | 차단 | 경고 | 통과(VEX) | 배포 판정 |
|---|---|---|---|---|---|---|---|
| `ai-serving:v1` | 126 (CRITICAL 26) | 65 (CRITICAL 8) | 146 | 5 | 126 | 15 | 불가 |
| `ai-serving:v2` | 44 (CRITICAL 0) | 53 (CRITICAL 4) | 61 | 0 | 53 | 8 | 가능 |

`[표 실습8-1]` 이미지 버전별 판정 결과 — 위 표가 이번 실행의 결과입니다.

베이스 이미지 한 줄과 의존성 버전을 바꾸는 것만으로 차단 건이 모두 사라졌습니다. 그러나 v2도 "깨끗한" 이미지는 아닙니다. 2025년에 고른 최신 버전도 2026년 10월에는 이미 HIGH 등급 CVE가 쌓여 있습니다. **오늘의 최신 버전은 내일의 오래된 버전**입니다. 그래서 8-3의 "저장소 재스캔"처럼 배포한 이미지도 매일 다시 대조하고, 영향받는 이미지는 자동으로 다시 빌드해야 합니다.

v2에 남은 경고가 어느 패키지에서 나오는지 봅니다.

In [16]:
table(["패키지", "경고 건수"], Counter(r["pkg"] for r in gate_rows("gate-v2.txt") if r["verdict"] == "경고").most_common(8))

| 패키지 | 경고 건수 |
|---|---|
| pillow | 39 |
| urllib3 | 6 |
| aiohttp | 5 |
| starlette | 3 |

## 6. 파이프라인에 넣기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 실습 앱과 두 가지 Dockerfile 살펴보기 | ✅ 완료 |
| 2 | v1 이미지 빌드 | ✅ 완료 |
| 3 | CVE 스캔과 SBOM 생성 | ✅ 완료 |
| 4 | KEV와 EPSS로 판정하기 | ✅ 완료 |
| 5 | v2로 다시 판정 | ✅ 완료 |
| 6 | 파이프라인에 넣기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

`gate.py`는 차단 건이 있으면 종료 코드 1을 돌려주므로, CI/CD의 한 단계로 넣으면 배포를 자동으로 막는 보안 Gate가 됩니다.

```yaml
# GitHub Actions 예시 단계
- name: Build image
  run: docker build -t ${{ env.IMAGE }} .
- name: Scan image
  run: trivy image --severity HIGH,CRITICAL --format json -o scan.json ${{ env.IMAGE }}
- name: SBOM
  run: trivy image --format cyclonedx -o sbom.json ${{ env.IMAGE }}
- name: Security gate (KEV/EPSS)
  run: python3 gate.py scan.json
```

파이프라인을 보안 Gate로 쓰려면 두 가지를 함께 정합니다.

- **머지를 막는 규칙으로 만듭니다**: Gate가 실패해도 코드를 합칠 수 있다면 경고에 그칩니다. 저장소 설정에서 이 단계가 성공해야만 기본 브랜치에 머지할 수 있게 합니다. 같은 파이프라인 첫 단계에 4장 실습 3의 Gitleaks를 넣으면 비밀정보 유출도 같은 방식으로 막습니다.
- **파이프라인이 쓰는 도구를 고정하고 검증합니다**: Trivy처럼 파이프라인 안에서 내려받는 도구는 `latest` 대신 특정 버전으로 고정하고, 내려받은 파일을 공식 체크섬과 대조합니다. 2026년 3월 Trivy 배포 과정 침해(5-2)처럼 도구의 배포 과정이 오염되면, 최신 버전을 자동으로 받는 파이프라인이 오염된 도구를 그대로 실행합니다.

## 7. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 실습 앱과 두 가지 Dockerfile 살펴보기 | ✅ 완료 |
| 2 | v1 이미지 빌드 | ✅ 완료 |
| 3 | CVE 스캔과 SBOM 생성 | ✅ 완료 |
| 4 | KEV와 EPSS로 판정하기 | ✅ 완료 |
| 5 | v2로 다시 판정 | ✅ 완료 |
| 6 | 파이프라인에 넣기 | ✅ 완료 |
| 7 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 8 | 정리 | 예정 |

### 7.1. 체크포인트

- [ ] 취약점의 대부분은 어디(베이스 이미지 / 직접 고른 라이브러리)에서 나왔나요? 이 결과는 오픈소스 선정 기준(8-2)에 무엇을 더해야 한다고 말해 주나요?
- [ ] CVSS 순서와 실제 위험(KEV·EPSS) 기준 판정이 달라진 CVE는 무엇이었나요?
- [ ] 수정 버전이 없는 CVE를 "통과(VEX)"로 처리하고 운영 모니터링(9-4)으로 넘기는 이유를 이해했나요?
- [ ] v2도 1년 뒤에는 v1처럼 될 것입니다. 이를 사람 손 없이 막으려면 파이프라인에 무엇이 더 필요할까요?

### 7.2. 과제 1: 판정 규칙 고쳐 보기

4.3에서 EPSS가 매우 높은데도 HIGH라서 "경고"에 머문 CVE가 있었습니다. 아래 `decide_v2()`를 고쳐 **HIGH라도 EPSS가 높으면 차단**하도록 바꾸고, 기준값(`HIGH_EPSS`)을 바꿔 가며 v1과 v2의 차단 건수가 어떻게 달라지는지 봅니다.

- 기준을 너무 낮추면 v2도 "배포 불가"가 될 수 있습니다. 그러면 무슨 일이 생길까요? (긴급 패치 배포까지 막히는 경우)
- 기준값을 정하는 사람과 예외를 승인하는 사람은 누구여야 할까요?

In [17]:
HIGH_EPSS = 0.5   # TODO: 바꿔 가며 시험합니다


def decide_v2(r):
    """gate.py의 decide()를 옮긴 것입니다. r은 gate_rows()의 한 줄입니다."""
    if r["kev"]:
        return "차단"
    if r["sev"] == "CRITICAL" and r["epss"] >= 0.1:
        return "차단"
    # TODO: HIGH 등급이고 EPSS가 HIGH_EPSS 이상이면 "차단"
    if r["fixed"] == "-":
        return "통과(VEX)"
    return "경고"


rows = []
for ver in ["v1", "v2"]:
    g = gate_rows(f"gate-{ver}.txt")
    before = Counter(r["verdict"] for r in g)
    after = Counter(decide_v2(r) for r in g)
    rows.append((ver, before["차단"], after["차단"], "불가" if after["차단"] else "가능"))
table(["이미지", "기존 규칙 차단", f"새 규칙 차단 (HIGH+EPSS≥{HIGH_EPSS})", "배포 판정"], rows)

| 이미지 | 기존 규칙 차단 | 새 규칙 차단 (HIGH+EPSS≥0.5) | 배포 판정 |
|---|---|---|---|
| v1 | 5 | 5 | 불가 |
| v2 | 0 | 0 | 가능 |

<details>
<summary>💡 힌트 보기</summary>

```python
    if r["sev"] == "HIGH" and r["epss"] >= HIGH_EPSS:
        return "차단"
```

`HIGH_EPSS`를 0.5, 0.2, 0.05로 바꿔 가며 v1과 v2의 차단 건수가 어떻게 변하는지 봅니다. 같은 방식으로 "CRITICAL + EPSS 0.1" 기준도 낮춰 보세요. 기준이 엄격할수록 위험은 줄지만, 고칠 수 없는(수정 버전이 없는) 취약점 때문에 배포가 멈추는 일도 늘어납니다. 그래서 차단 기준과 함께 **예외 승인 절차**(누가, 얼마 동안, 어떤 보완 통제를 조건으로)를 정해 둡니다.
</details>

### 7.3. 과제 2: 배포 뒤의 재스캔 설계 (서술)

v2는 오늘 "배포 가능"이었지만, 내일 새 CVE가 KEV에 등재되면 차단 대상이 됩니다. 아래 질문에 답해 보세요.

1. 이미 배포된 이미지를 매일 다시 판정하려면 무엇을 입력으로 써야 할까요? (이미지를 다시 스캔? 3단계의 SBOM을 새 취약점 DB와 대조?)
2. 재판정에서 "차단"이 나오면 누구에게 알리고, 몇 시간 안에 무엇을 해야 할까요? (9장의 패치 주기와 연결)

✍️ **답 적기** (이 셀을 더블클릭해 고칩니다)

1.
2.

## 8. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 실습 앱과 두 가지 Dockerfile 살펴보기 | ✅ 완료 |
| 2 | v1 이미지 빌드 | ✅ 완료 |
| 3 | CVE 스캔과 SBOM 생성 | ✅ 완료 |
| 4 | KEV와 EPSS로 판정하기 | ✅ 완료 |
| 5 | v2로 다시 판정 | ✅ 완료 |
| 6 | 파이프라인에 넣기 | ✅ 완료 |
| 7 | 체크포인트와 과제 | ✅ 완료 |
| 8 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

In [18]:
%%bash
docker rmi ai-serving:v1 ai-serving:v2 python:3.9.0-slim python:3.12-slim >/dev/null 2>&1; echo "실습 이미지 삭제됨"
rm -rf ~/image-lab && echo "실습 폴더 삭제됨"

실습 이미지 삭제됨


실습 폴더 삭제됨


---
> 다음 실습 2에서는 이미지가 아닌 **모델**을 시험합니다. 같은 보안 Gate 원리를 "공격 성공률"에 적용합니다.